In [2]:
import pandas as pd
import numpy as np

# v2 파일 경로 직접 지정
FILE_PATH = r"final_merged(단칼)_v2.xlsx"
df = pd.read_excel(FILE_PATH)
df['WATCH_DAY'] = pd.to_datetime(df['WATCH_DAY'])

# ── 중간 계산 (저장 안 함) ──────────────────────────
df['days_since_reg'] = (df['WATCH_DAY'] - df['reg_date']).dt.days
df = df[df['days_since_reg'] >= 0]

def assign_week(d):
    if d <= 7:   return 'w1'
    elif d <= 14: return 'w2'
    elif d <= 21: return 'w3'
    else:         return 'w4'

df['week_bucket'] = df['days_since_reg'].apply(assign_week)

# ── 주차별 시청 시간 집계 ───────────────────────────
weekly = df.groupby(['USER_NUM', 'week_bucket'])['DURATION'].sum().unstack(fill_value=0)
weekly.columns = ['(파생)duration_' + c for c in weekly.columns]

for col in ['(파생)duration_w1', '(파생)duration_w2', '(파생)duration_w3']:
    if col not in weekly.columns:
        weekly[col] = 0

weekly['(파생)duration_w4'] = np.nan
weekly = weekly[['(파생)duration_w1', '(파생)duration_w2',
                 '(파생)duration_w3', '(파생)duration_w4']]

weekly['(파생)duration_w1_to_w2_ratio'] = (
    (weekly['(파생)duration_w2'] - weekly['(파생)duration_w1']) /
    (weekly['(파생)duration_w1'] + 1)
)
weekly['(파생)duration_w1_to_w3_ratio'] = (
    (weekly['(파생)duration_w3'] - weekly['(파생)duration_w1']) /
    (weekly['(파생)duration_w1'] + 1)
)
weekly['(파생)duration_w1_to_w4_ratio'] = np.nan
weekly['(파생)duration_w2_to_w4_ratio'] = np.nan
weekly['(파생)duration_w3_to_w4_ratio'] = np.nan

# ── 사용자 단위 집계 ────────────────────────────────
agg_first = {
    'gender': 'first',
    'age': 'first',
    'product_cd': 'first',
    'amount': 'first',
    'billing_method': 'first',
    'concurrent_streams': 'first',
    'promotion_yn': 'first',
    'is_user_verified': 'first',
    'reg_hour': 'first',
    '(파생)duration_days': 'first',
    'repurchase': 'first',
    'is_churn_prevented': 'first',
}

agg_custom = {
    'is_weekend': 'mean',
    'DURATION': 'sum',
    'MOVIE_NUM': 'nunique',
    'WATCH_DAY': 'nunique',
}

user_df = df.groupby('USER_NUM').agg({**agg_first, **agg_custom}).reset_index()

# 컬럼명 정리
user_df = user_df.rename(columns={
    'is_weekend': '(파생)weekend_ratio',
    'DURATION': '(파생)total_duration',
    'MOVIE_NUM': '(파생)unique_movies',
    'WATCH_DAY': '(파생)active_days',
})

# ── 주차 파생변수 합류 ──────────────────────────────
user_df = user_df.merge(weekly, on='USER_NUM', how='left')

# ── 확인 ────────────────────────────────────────────
print('shape:', user_df.shape)
print()
print(user_df.head(5).T)
print()
print('NaN 현황:')
print(user_df.isnull().sum())

# ── 저장 ────────────────────────────────────────────
output_path = r"final_merged_user(단칼)_v1.xlsx"
user_df.to_excel(output_path, index=False)
print(f'\n저장 완료: {output_path}')

shape: (14461, 26)

                                    0        1         2         3        4
USER_NUM                            0        1         2         4        5
gender                              N        M         F         N        M
age                                40       40        30        40       30
product_cd                    pk_1489  pk_1487   pk_1487   pk_1508  pk_1506
amount                        13900.0    100.0     100.0      9.99    13.49
billing_method                    134      134       190       140      140
concurrent_streams                  4        1         1         1        2
promotion_yn                        0        1         1         0        0
is_user_verified                    0        1         1         0        1
reg_hour                           12       16        23        12       20
(파생)duration_days                  31       31        31        32       32
repurchase                          1        0         1         0  